# Atmospheric restart stability checks

Discovers native EAM restart files across ensemble members, computes cached physical-range and non-finite checks, measures changes between snapshots, and plots member summaries plus spatial flag maps.


In [ ]:
from pathlib import Path
INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")

case_name = "DARTEN10_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy"
time_tags = ("2011-12-24-00000", "2011-12-25-00000")
variable_requests = (
    {"name": "T", "level": 5, "dim": "lev", "valid_min": 100.0, "valid_max": 400.0},
    {"name": "Q", "level": 79, "dim": "lev", "valid_min": 0.0, "valid_max": 0.1},
    {"name": "U", "level": 79, "dim": "lev", "valid_min": -250.0, "valid_max": 250.0},
    {"name": "V", "level": 79, "dim": "lev", "valid_min": -250.0, "valid_max": 250.0},
    {"name": "PS", "level": None, "dim": None, "valid_min": 10000.0, "valid_max": 120000.0},
)
ensemble_ids = None  # for example ("EN01", "EN02") for a quick subset
map_requests = ("T_lev05",)
map_snapshots = None  # None plots every selected timestamp
maximum_allowed_flagged_fraction = 0.0
fail_on_flags = False
force_compute = False
show = True

In [ ]:
import sys
import matplotlib.pyplot as plt
from IPython.display import display
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))

from configs.output_paths import workflow_output_dirs
from util.atm_restart_stability import (
    AVAILABLE_RESTART_VARIABLES,
    AVAILABLE_METRICS,
    RestartVariableSpec,
    restart_stability_cache_path,
    cache_restart_stability,
    plot_restart_summary,
    plot_flag_fraction_maps,
)


## Notes: available variables and metrics

Native EAM restart variables supported by the default catalog are:

- `PS`: surface pressure; suggested bounds 10,000–120,000 Pa.
- `T`: air temperature on `lev`; suggested bounds 100–400 K.
- `Q`: specific humidity on `lev`; suggested bounds 0–0.1 kg kg⁻¹.
- `U` and `V`: winds on `lev`; suggested bounds −250–250 m s⁻¹.

Bounds are configurable screening limits, not universal scientific validity limits. Add another numeric `ncol` variable by supplying its name, optional level, dimension, and appropriate bounds.

Cached member metrics are `minimum`, `maximum`, `mean`, `standard_deviation`, `nan_count`, `inf_count`, `below_range_count`, `above_range_count`, `total_count`, and `flagged_fraction`. Spatial products are `out_of_range_fraction_map`, `ensemble_mean`, `ensemble_minimum`, and `ensemble_maximum`. Transition metrics are `rms_change` and `maximum_absolute_change` between consecutive snapshots.

The cache is request-keyed and reused unless `force_compute = True`. Set `fail_on_flags = True` to turn the notebook into a strict validation gate.


In [ ]:
requests = tuple(
    RestartVariableSpec(
        item["name"], item["level"], item["dim"],
        item["valid_min"], item["valid_max"],
    )
    for item in variable_requests
)
data_dir, figure_dir = workflow_output_dirs(
    "analysis_atm_init", "restart_stability", output_root=DIAGNOSTIC_OUTPUT_ROOT
)
cache_path = restart_stability_cache_path(
    data_dir, case_name, time_tags, requests, ensemble_ids=ensemble_ids
)
existed = cache_path.is_file()
diagnostics = cache_restart_stability(
    cache_path,
    case_dir=INPUT_DATA_ROOT / case_name,
    time_tags=time_tags,
    requests=requests,
    ensemble_ids=ensemble_ids,
    force_compute=force_compute,
)
print("[REUSE]" if existed and not force_compute else "[WRITE]", cache_path)

summary_columns = (
    "minimum", "maximum", "mean", "standard_deviation",
    "nan_count", "inf_count", "below_range_count",
    "above_range_count", "flagged_fraction",
)
display(diagnostics[list(summary_columns)].to_dataframe().dropna(how="all"))

worst = float(diagnostics.flagged_fraction.max(skipna=True))
print(f"Maximum flagged fraction: {worst:.6g}")
if fail_on_flags and worst > maximum_allowed_flagged_fraction:
    raise RuntimeError(
        f"Restart stability check failed: {worst:.6g} exceeds "
        f"{maximum_allowed_flagged_fraction:.6g}"
    )

fig = plot_restart_summary(diagnostics, show=show)
summary_figure = figure_dir / f"{cache_path.stem}_summary.png"
fig.savefig(summary_figure, dpi=200, bbox_inches="tight")
if not show:
    plt.close(fig)
print("[FIGURE]", summary_figure)

if map_requests:
    fig = plot_flag_fraction_maps(
        diagnostics,
        requests=map_requests,
        snapshots=map_snapshots,
        show=show,
    )
    map_figure = figure_dir / f"{cache_path.stem}_flag_maps.png"
    fig.savefig(map_figure, dpi=200, bbox_inches="tight")
    if not show:
        plt.close(fig)
    print("[FIGURE]", map_figure)


## Inter-snapshot changes

The following table reports per-member RMS and maximum absolute changes between consecutive restart snapshots. These are diagnostic magnitudes; interpretation depends on the variable, level, and time separation.


In [ ]:
if diagnostics.sizes.get("transition", 0):
    display(
        diagnostics[["rms_change", "maximum_absolute_change"]]
        .to_dataframe()
        .dropna(how="all")
    )
else:
    print("Select at least two timestamps to compute transition metrics.")
